## EDA before recursivecharatertextsplitter 
I want to controll if there are **strange characters** or **noise** from the scraping , i also want to
identify unnecessary texts i.e **menues** or **javascript warnings** 
i also want to see the **paragraph division** so i can choose the best `chunk size`


In [ ]:
with open ("backend/data/utlandsstudier/csn_utlandsstudier.txt", "r", encoding="utf-8") as f:
    text = f.read()


In [11]:
# Looking for "javascript phrases"

javascript_keywords = ["javascript", ",", "function", "var", "const"]

javascript_garbage = [
    line for line in text.split('\n')
    if any(key in line.lower() for key in javascript_keywords)
]

print(f" Ahaa look at that ! i found {len(javascript_garbage)} rows with potential Javascript garbage:")

for line in javascript_garbage[:5]:  
    print(f"-{line}")





 Ahaa look at that ! i found 70 rows with potential Javascript garbage:
-Du kan ansöka om studiemedel när du ska studera utomlands på eftergymnasial nivå. Du som är över 20 år kan även ansöka om studiemedel för studier på gymnasienivå, till exempel språkkurser. Studiemedel består av bidrag och lån.
-Se alla belopp för deltid och heltid vid utlandsstudier, 2026 Pdf, 47 kB. (Pdf, 47 kB)
-Se alla belopp för deltid och heltid för utlandsstudier, 2025 Pdf, 71 kB. (Pdf, 71 kB)
-Det finns även fler bidrag och lån som du kan ansöka om i olika situationer, bland annat merkostnadslån för undervisningsavgift (tuition fee), resa och försäkring. Vissa lån kan du bara få om du studerar inom EU/EES och Schweiz, till exempel tilläggslån.
-Den här funktionen kräver att du har javascript.


Cleaning step: Create a function to filter out JavaScript-warnings, PDF-links, and empty rows to ensure the chatbot gives relevant and factual content !

Examples of **noises** found in the previous output 

"Den här funktionen kräver att du har javascript"  and  "Pdf, 47 kB" 


In [17]:
## LLM USAGE, here i have used assistance from LLM to make sure the cleaning process is done correctly
import re 

def clean_csntext(raw_text):
    lines = raw_text.split('\n')
    cleaned_lines = []

    for line in lines:
        l = line.strip()

        # 1. Skip empty rows
        if not l:
            continue 

        # 2. Skip javascript rows
        if "kräver att du har javascript" in l.lower():
            continue 

        # 3. Skip template-variables {{...}}
        if "{{" in l and "}}" in l:
            continue

       
        cleaned_lines.append(l)

    
    return "\n\n".join(cleaned_lines)


clean_Text = clean_csntext(text)


print(clean_Text[:500])

Vi gör studier möjligt.

Du kan ansöka om studiemedel när du ska studera utomlands på eftergymnasial nivå. Du som är över 20 år kan även ansöka om studiemedel för studier på gymnasienivå, till exempel språkkurser. Studiemedel består av bidrag och lån.

Beloppet för bidraget och studielånet är samma oavsett om du studerar i Sverige eller utomlands. När du studerar utomlands kan du även ta ett extra lån för ökade levnadskostnader. Det kallas merkostnadslån för utlandsstudier.

Se alla belopp för d


This looks much better !, however you can see that the output stops at " Se alla belopp för d " 
im checking the csn_utlandsstudier.txt to see if there are columns or numbers further down the text , if there is only " se belopp i pdf " this means that we are missing some important details.

In [18]:
# Print characters 500 to 200 to see what comes after the introduction 
print(clean_Text[500:2000])


eltid och heltid vid utlandsstudier, 2026 Pdf, 47 kB. (Pdf, 47 kB)

Se alla belopp för deltid och heltid för utlandsstudier, 2025 Pdf, 71 kB. (Pdf, 71 kB)

Det finns även fler bidrag och lån som du kan ansöka om i olika situationer, bland annat merkostnadslån för undervisningsavgift (tuition fee), resa och försäkring. Vissa lån kan du bara få om du studerar inom EU/EES och Schweiz, till exempel tilläggslån.

Merkostnadslån för dig med extra kostnader

Tilläggsbidrag för dig som har barn

Tilläggslån för dig som har arbetat tidigare

Använd uträknaren för att räkna ut ungefär hur mycket du kan få och hur återbetalningen kan bli. Beloppen gäller för 2026.

Totalt belopp

Lån

Bidrag

Summa

Bidrag och lån under studietiden

Bidrag

Tilläggsbidrag för barn

Studielån

Merkostnadslån för utlandsstudier

Tilläggslån

Merkostnadslån

Resa

Försäkring

Undervisningsavgift

Kriterier

Land

Veckor med studiemedel

Kalenderhalvår

Studietakt

Studienivå

Totalkostnad

Lån

Ränta

Expeditionsavg

# Missing data , More PDF traps = More to clean !
# Cleaning Version 2: Removing PDF-links and short noise.


In [19]:
#LLM USAGE, here i have used assistance from LLM to make sure the cleaning process is done correctly

import re 

def clean_csntext_v2(raw_text):
    lines = raw_text.split('\n')
    cleaned_lines = []

    for line in lines:
        l = line.strip()

        #. 1 Empty rows 
        if not 1:
            continue


        # 2. Javascript-warnings 
        if "kräver att du har javascript" in l.lower():
            continue

        # 3. PDF referenses new filter 
        if "pdf," in l.lower() or "(pdf," in l.lower():
            continue 

        # 4. Rows / noises new added filter !
        if len (l) <10 and l not in ["Kriterier", "Land"]:
            continue 

        # 5. Template-variables 
        if "{{" in l and "}}" in l:
            continue
 

        
        cleaned_lines.append(l)

    return "\n\n".join(cleaned_lines)


clean_Text_v2 = clean_csntext_v2(text)


print(clean_Text_v2[500:1500])





bidrag och lån som du kan ansöka om i olika situationer, bland annat merkostnadslån för undervisningsavgift (tuition fee), resa och försäkring. Vissa lån kan du bara få om du studerar inom EU/EES och Schweiz, till exempel tilläggslån.

Merkostnadslån för dig med extra kostnader

Tilläggsbidrag för dig som har barn

Tilläggslån för dig som har arbetat tidigare

Använd uträknaren för att räkna ut ungefär hur mycket du kan få och hur återbetalningen kan bli. Beloppen gäller för 2026.

Totalt belopp

Bidrag och lån under studietiden

Tilläggsbidrag för barn

Merkostnadslån för utlandsstudier

Tilläggslån

Merkostnadslån

Försäkring

Undervisningsavgift

Kriterier

Land

Veckor med studiemedel

Kalenderhalvår

Studietakt

Studienivå

Totalkostnad

Expeditionsavgift

Snitt per månad

Årlig kostnad

Ansök om studiemedel

Du ansöker om studiemedel på Mina sidor, snabbast och smidigast går det om du har en e-legitimation. När du ska studera utomlands måste du skicka med bilagor i din ansökan. O

This looks much better i can see that most of the missing data is gone and PDF traps gone as well as noises are gone , however i can see that the next sentence is missing some information 
Also there are still some labels left such as "Merkostnadslån för utlandsstudier" " Bidrag och lån under studietiden" , i will keep the headers so that it can provide the chatbot with context about what types of loans that exists even though it cant provide exact amount.

Also i can see that the text was cut in the last part so i will also adjust that in the next cell

In [20]:
print(clean_Text_v2[1500:3000])

m det saknas nödvändiga bilagor kan vi inte fatta beslut om din ansökan. Då kontaktar vi dig och ber dig att skicka in det som saknas. Det innebär att det tar längre tid innan du får ditt beslut.

Se vilka bilagor som du kan behöva skicka med:

Du behöver skicka med ett officiellt antagningsbevis (acceptance letter) första gången du ansöker om studiemedel. På antagningsbeviset ska det stå:

Om du tänkt ansöka om merkostnadslån för undervisningsavgift (tuition fee) så måste du styrka kostnaden med ett personligt intyg från skolan. Tänk på att allmän information från skolans webbplats inte godkänns av CSN, intyget måste vara personligt skrivit till dig. På intyget ska det stå:

Om studietiderna inte står på antagningsbeviset så måste du skicka in en separat bilaga från skolan. På dokumentet ska det stå:

Det går bra att skicka in en kopia på den akademiska kalendern från skolans webbplats.

Vid vissa utbildningar har CSN redan bestämt vilka terminstider man kan få studiemedel för, det ka

Its apparent that it removed crucial information as "acceptance letter" , "tuition fee", "5509w"


In [21]:
# Check total statistic after cleaning 

total_characters= len(clean_Text_v2)
total_amount_of_words = len(clean_Text_v2.split())


print(f"---Statistic for cleaned text---")
print(f"total characters: {total_characters}")
print (f" total amount of words: {total_amount_of_words}")

print(f"Approximate amount of tokens ( word * 1.3): {int(total_amount_of_words* 1.3)}")




---Statistic for cleaned text---
total characters: 25095
 total amount of words: 4056
Approximate amount of tokens ( word * 1.3): 5272


In [32]:
# Last step before chunking , to look for unnecessary characters like --- which is a  waste of tokens 
inspection_start = 10000
print(clean_Text_v2[inspection_start: inspection_start+ 1500])





tanför EU/EES och Schweiz så måste dina studier pågå i minst 13 veckor på heltid. Du kan aldrig få studiemedel för studier på mindre än 50 procents studietakt. CSN kan göra undantag från kravet på heltidsstudier om du har särskilda skäl, exempelvis om du bara har några få kurser kvar av din utbildning eller om du blir sjuk.

Studietakt

Om du har fått studiemedel innan så måste du ha klarat tillräckligt med poäng och kurser för att kunna få nya studiemedel. När du ansöker om studiemedel kontrollerar vi att du har klarat tillräckliga studieresultat.

Studieresultat

Med distansstudier menar vi att du studerar på en utbildning vid en utländsk skola där du helt eller delvis inte läser face-to-face med lärare i klassrum, utan studerar hemifrån. Det gäller även om du befinner dig i studielandet eller på campus men läser dina kurser online.

Du kan få studiemedel för distansstudier inom EU/EES och i Schweiz.

I länder utanför EU/EES och i Schweiz kan du bara få studiemedel för distansstudier

In [34]:

with open("backend/data/utlandsstudier/csn_utlandsstudier.txt", "w", encoding="utf-8") as f:
    f.write(clean_Text_v2)

print("file succesfully saved in  backend/data/utlandsstudier/csn_utlandsstudier.txt")

file succesfully saved in  backend/data/utlandsstudier/csn_utlandsstudier.txt
